In [1]:
from functools import partial
import os

#os.environ["LIBTPU_INIT_ARGS"] = "--xla_jf_dump_to=/tmp/llo --xla_jf_dump_llo_proto=true"
#os.environ["XLA_FLAGS"] = "--xla_jf_dump_to=/tmp/llo --xla_jf_dump_llo_proto=true"
#os.environ["LIBTPU_INIT_ARGS"] = "--xla_jf_dump_to=/tmp/llo --xla_jf_dump_llo_proto=true"
os.environ["LIBTPU_INIT_ARGS"] = " ".join([
  "--xla_tpu_use_tc_device_shape_on_sc=true",
  # "--xla_tpu_enable_offloading_scatter_to_sparsecore=true",
])

import jax
import jax.numpy as jnp
from jax import random
from jax.experimental import pallas as pl
import jax.experimental.pallas.tpu as pltpu
import jax.experimental.pallas.tpu_sc as plsc
from jax.sharding import PartitionSpec as P

import tune_jax
jax.devices()

[TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0),
 TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0),
 TpuDevice(id=2, process_index=0, coords=(0,1,0), core_on_chip=0),
 TpuDevice(id=3, process_index=0, coords=(1,1,0), core_on_chip=0)]

# Mesh

In [ ]:
mesh = jax.make_mesh((jax.device_count(),), ("x",), axis_types=jax.sharding.AxisType.Explicit)
keys = iter(random.split(random.key(0), 1024))
jax.sharding.set_mesh(mesh)
x = jax.jit(lambda: random.normal(next(keys), (1024,)), out_shardings=P("x"))()
jax.typeof(x)

# modified kernel

In [ ]:
num_steps, x_cols = 8 * 8 * 4096, 7168
x = jnp.arange(num_steps * x_cols).reshape(num_steps, x_cols)  # random data
indices = jnp.arange(x.shape[0])

In [ ]:
# n = 8 * 4096
# keys = iter(random.split(random.key(0), 1024))
# x = random.normal(next(keys), (n, 56, 128), dtype=jnp.bfloat16)
# idx = jnp.argsort(random.normal(next(keys), (n,)))

In [ ]:
@partial(jax.jit, static_argnames=("block_size",))
def gather_pallas(x, idx, block_size: int = 128):
  def gather_kernel(x_ref, idx_ref, out_ref, sem_refs, idx_scratch):
    idx0 = pl.program_id(0)
    pltpu.sync_copy(idx_ref.at[idx0, 0, :], idx_scratch)
    copies = [pltpu.async_copy(
      x_ref.at[idx_scratch[i], ...], out_ref.at[idx0 * block_size + i, ...], sem_refs.at[i]
    ) for i in range(block_size)]
    [copy.wait() for copy in copies]

  grid = pl.cdiv(n, block_size)
  out = pl.pallas_call(
    gather_kernel,
    grid=grid,
    in_specs=[pl.BlockSpec(memory_space=pltpu.MemorySpace.ANY), pl.BlockSpec(memory_space=pltpu.MemorySpace.ANY)],
    out_specs=pl.BlockSpec(memory_space=pltpu.MemorySpace.ANY),
    out_shape=jax.ShapeDtypeStruct((idx.shape[0],) + x.shape[1:], dtype=x.dtype),
    scratch_shapes=[pltpu.SemaphoreType.DMA(block_size,), pltpu.MemorySpace.SMEM(block_size, jnp.int32)]
  )(x, idx.reshape((-1, 1, block_size)))
  return out

In [ ]:
1e-3 / (2 * x.size / 1640e9)

In [ ]:
block_size = 128
out = gather_pallas(x, idx, block_size=block_size)
gather = jax.jit(lambda x, idx: x[idx, ...])
_ = gather(x, idx)
with jax.profiler.trace("/tmp/gather"):
  for _ in range(3):
    out = jax.block_until_ready(gather_pallas(x, idx, block_size=block_size))
  for _ in range(3):
    out = jax.block_until_ready(gather(x, idx))

In [ ]:
round_to = lambda x, r: pl.cdiv(max(x, r), r) * r

def my_fun(indices, x, row, col):
  #row_window, col_window = round_to(min(row, num_steps), 16), round_to(min(x_cols, col), 128)
  row_window, col_window = round_to(min(row, num_steps), 16), round_to(min(x_cols, col), 8)

  def kernel(indices_ref, x_ref, o_ref):
    del indices_ref  # only used by the memory pipeline
    chunk = 16  # hardware specific value
    @pl.loop(0, x_ref.shape[-1], step=chunk)
    def _(i):
      #o_ref[:, pl.ds(i, chunk)] = x_ref[:, pl.ds(i, chunk)]

      @pl.loop(0, x_ref.shape[1], step=4)
      def _(j):
        @pl.loop(0, x_ref.shape[0], step=1)
        def _(k):
          #o_ref[pl.ds(j, 4), k, pl.ds(i, chunk)] = x_ref[pl.ds(j, 4), k, pl.ds(i, chunk)]
          o_ref[k, pl.ds(j, 4), pl.ds(i, chunk)] = x_ref[k, pl.ds(j, 4), pl.ds(i, chunk)]

  out_shape = x
  grid = (pl.cdiv(indices.size, row_window), pl.cdiv(x.shape[-1], col_window))
  #print(f"{grid = } {row_window = } {col_window = }", flush=True)

  in_specs = [
    pl.BlockSpec((row_window,), lambda i, j: i),
    # this indexed by performs the gather inside the memory pipeline itself
    plsc.BlockSpec((row_window, col_window, x.shape[-1]), lambda i, j: (0, j, 0), indexed_by=0, indexed_dim=0)
  ]
  out_specs = pl.BlockSpec((row_window, col_window, x.shape[-1]), lambda i, j: (i, j, 0))

  pallas_gather = pl.pallas_call(
    kernel,
    out_shape=out_shape, grid=grid, in_specs=in_specs, out_specs=out_specs,
    compiler_params=pltpu.CompilerParams(dimension_semantics=("parallel", "arbitrary"), kernel_type=pltpu.KernelType.SC_VECTOR_SUBCORE),
  )
  out = pallas_gather(indices, x)
  return out


# %%
tune_jax.logger.setLevel("INFO")

hyperparams = {
  "row": [16, 32, 64, 128],
  "col": [128, 256, 512, 1024],
}
my_fun(indices, x, row=16, col=4)

In [ ]:
fn = tune_jax.tune(my_fun, hyperparams=hyperparams)
fn(indices, x)
print(tune_jax.tabulate(fn))
fn2 = tune_jax.tune(lambda idx, x: x[idx, :])
fn2(indices, x)
print(tune_jax.tabulate(fn2))

with jax.profiler.trace("/tmp/gather"):
  for _ in range(3):
    jax.block_until_ready(fn(indices, x))
  for _ in range(3):
    jax.block_until_ready(fn2(indices, x))

# Async kernel

In [18]:
num_steps, x_cols = 8 * 8 * 4096, 7168
x = jnp.arange(num_steps * x_cols, dtype=jnp.float32).reshape(num_steps, x_cols)  # random data
indices = jnp.arange(x.shape[0])

@jax.jit
def my_fun(indices, x):
  LANES = 128
  if x.ndim != 3: # for testing
    x = x.reshape((x.shape[0], -1, LANES))  # make sure x is in [B, E // LANES, LANES]

  def kernel(indices_ref, x_ref, o_ref):
    @pl.loop(0, indices.shape[0], step=8)
    def _(i):
      i = (i // 8) * 8
      slice = pl.ds(i, 8)
      pltpu.sync_copy(x_ref.at[indices_ref[slice]].reshape((slice.size, o_ref.shape[-1])), o_ref.at[slice, :])

  out_shape = jax.ShapeDtypeStruct((x.shape[0], x.shape[-2] * x.shape[-1]), x.dtype)
  grid = (pl.cdiv(x.shape[1], 8),)

  in_specs = [
    pl.BlockSpec(memory_space=pltpu.MemorySpace.VMEM),
    pl.BlockSpec(memory_space=pltpu.MemorySpace.HBM),
  ]
  out_specs = pl.BlockSpec(memory_space=pltpu.MemorySpace.HBM)

  sc_copy = pl.pallas_call(
    kernel,
    out_shape=out_shape, grid=grid, in_specs=in_specs, out_specs=out_specs,
    compiler_params=pltpu.CompilerParams(kernel_type=pltpu.KernelType.SC_VECTOR_SUBCORE),
  )
  return sc_copy(indices, x)

out = my_fun(indices, x)

ValueError: Reshape a ref with different number of elements: (56, 128) vs (8, 7168)

In [39]:
@jax.jit
def my_fun(indices, x):
  LANES = 128
  if x.ndim != 3:
    x = x.reshape((x.shape[0], -1, LANES))  # make sure x is in [B, E // LANES, LANES]

  grid = (8,)
  steps_per_grid_entry = pl.cdiv(x.shape[0], grid[0])

  #def kernel(indices_ref, x_ref, o_ref, x_scratch, o_scratch):
  def kernel(x_ref, o_ref, x_scratch, o_scratch):
    chunk = 16  # hardware specific value
    idx = pl.program_id(0)
    
    # scratch_bs = x_scratch.shape[0]
    scratch_bs = 1
    
    @pl.loop(idx * steps_per_grid_entry, (idx + 1) * steps_per_grid_entry, step=scratch_bs)
    def _(outer_idx):
      # outer_idx = pl.multiple_of(outer_idx, 8)
      # pltpu.sync_copy(x_ref.at[pl.ds(outer_idx, scratch_bs), ...], x_scratch)
      # pltpu.sync_copy(x_ref.at[outer_idx, ...], x_scratch)
      
      # pltpu.sync_copy(x_ref.at[outer_idx, ...], o_ref.at[outer_idx, ...].reshape((x_ref.shape[1:])))
      pltpu.sync_copy(x_ref.at[outer_idx, ...], x_scratch)
      
      chunk = 8  # hardware specific
      
      @pl.loop(0, o_ref.shape[0], step=1)
      def _(i):
        @pl.loop(0, x_ref.shape[0], step=4)
        def _(j):
          @pl.loop(0, x_ref.shape[1], step=chunk)
          def _(k):
            j_ = j * 4
            o_scratch[i, pl.ds(j * 4 * chunk + k, 4 * chunk)] = x_scratch[pl.ds(j, 4), pl.ds(k, chunk)].reshape(-1)

      # @pl.loop(0, o_ref.shape[-1], step=chunk)
      # def _(j):
      #   j_, k_=  j // (x.shape[-1] // chunk), jax.lax.rem(j, (LANES // chunk))
      #   o_scratch[pl.ds(j, chunk)] = 2 * x_scratch[j_, pl.ds(k_, chunk)]

      # pltpu.sync_copy(o_scratch, o_ref.at[pl.ds(outer_idx, scratch_bs), ...])
      pltpu.sync_copy(o_scratch, o_ref.at[pl.ds((outer_idx // 8) * 8, 8), ...])

  out_shape = jax.ShapeDtypeStruct((x.shape[0], x.shape[-2] * x.shape[-1]), x.dtype)

  in_specs = [
    #pl.BlockSpec((indices.shape[0] // grid[0],), lambda i: (i,), memory_space=pltpu.MemorySpace.SMEM),
    pl.BlockSpec(memory_space=pltpu.MemorySpace.HBM),
  ]
  out_specs = pl.BlockSpec(memory_space=pltpu.MemorySpace.HBM)

  sc_copy = pl.pallas_call(
    kernel,
    out_shape=out_shape, grid=grid, in_specs=in_specs, out_specs=out_specs,
    compiler_params=pltpu.CompilerParams(kernel_type=pltpu.KernelType.SC_VECTOR_SUBCORE),
    scratch_shapes=[
      pltpu.VMEM((x.shape[-2], x.shape[-1]), x.dtype),   # x_scratch
      pltpu.VMEM((8, x.shape[-1] * x.shape[-2]), x.dtype),  # o_scratch
    ]
  )
  #return sc_copy(indices, x)
  return sc_copy(x)

In [40]:
out = my_fun(indices, x)

JaxRuntimeError: INTERNAL: Failed to run pass pipeline.my_fun.5:112:1: error: failed to legalize operation 'vector.insert_strided_slice'
my_fun.5:112:1: note: see current operation: %372 = "vector.insert_strided_slice"(%371, %25) <{offsets = [0, 0], strides = [1, 1]}> : (vector<1x8xf32>, vector<4x8xf32>) -> vector<4x8xf32>


In [28]:
out

Array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)

# Actual kernel

In [19]:
# %%
num_steps, x_cols = 8 * 8 * 4096, 7168
x = jnp.arange(num_steps * x_cols).reshape(num_steps, x_cols)  # random data
indices = jnp.arange(x.shape[0])

In [20]:
@partial(jax.jit, static_argnames=("row", "col"))
def my_fun(indices, x, row, col):
  x = x.reshape((x.shape[0], -1, 128))
  #row_window, col_window = round_to(min(row, num_steps), 16), col

  def kernel(indices_ref, x_ref, o_ref, o_scratch, x_scratch):
    chunk = 16  # hardware specific value
    pltpu.sync_copy(x_ref.at[pl.ds(0, 8), ...], x_scratch)
    # o_scratch[...] = x_scratch.reshape(o_scratch.shape)[...]
    @pl.loop(0, o_ref.shape[0], step=4)
    def _(i):
      @pl.loop(0, o_ref.shape[-1], step=16)
      def _(j):
        j_, k_=  j // (128 // 16), jax.lax.rem(j, (128 // 16))
        # o_scratch[pl.ds(i, 4), pl.ds(j, 16)] = x_scratch[pl.ds(i, 4), pl.ds(j, 1), pl.ds(k_, 16)].reshape((4, 16))
        #o_scratch[pl.ds(i, 4), pl.ds(j, 16)] = x_scratch[0, pl.ds(i, 4), pl.ds(k_, 16)].reshape((4, 16))
        o_scratch[i, pl.ds(j, 16)] = 2 * x_scratch[0, j, pl.ds(i, 16)]
    pltpu.sync_copy(o_scratch, o_ref.at[pl.ds(0, 8), ...])

  out_shape = jax.ShapeDtypeStruct((x.shape[0], x.shape[-2] * x.shape[-1]), x.dtype)
  grid = (8,)

  in_specs = [
    pl.BlockSpec((indices.shape[0] // grid[0],), lambda i: (i,), memory_space=pltpu.MemorySpace.SMEM),
    pl.BlockSpec(memory_space=pltpu.MemorySpace.ANY),
    #plsc.BlockSpec((row_window, col_window, x.shape[-1]), lambda i, j: (0, j, 0), indexed_by=0, indexed_dim=0)
    #pl.BlockSpec((row_window, x.shape[1], x.shape[2]), lambda i: (i, 0, 0)),
  ]
  # out_specs = pl.BlockSpec((row_window, col_window,), lambda i, j: (i, j))
  # out_specs = plsc.BlockSpec((row_window, out_shape.shape[-1],), lambda i: (0, 0), indexed_by=0, indexed_dim=0)
  out_specs = pl.BlockSpec(memory_space=pltpu.MemorySpace.ANY)

  pallas_gather = pl.pallas_call(
    kernel,
    out_shape=out_shape, grid=grid, in_specs=in_specs, out_specs=out_specs,
    #compiler_params=pltpu.CompilerParams(dimension_semantics=("parallel", "arbitrary"), kernel_type=pltpu.KernelType.SC_VECTOR_SUBCORE),
    #compiler_params=pltpu.CompilerParams(dimension_semantics=("parallel",), kernel_type=pltpu.KernelType.SC_VECTOR_SUBCORE),
    compiler_params=pltpu.CompilerParams(kernel_type=pltpu.KernelType.SC_VECTOR_SUBCORE),
    scratch_shapes=[
      pltpu.VMEM((8, x.shape[-1] * x.shape[-2]), x.dtype),
      pltpu.VMEM((8, x.shape[-2], x.shape[-1]), x.dtype)
    ]
  )
  out = pallas_gather(indices, x)
  return out
  #assert jnp.linalg.norm(out - x[indices, :]) < 1e-3

In [2]:
num_steps, x_cols = 8 * 8 * 4096, 7168
x = jnp.arange(num_steps * x_cols).reshape(num_steps, x_cols)  # random data
indices = jnp.arange(x.shape[0])

@partial(jax.jit, static_argnames=("row", "col"))
def my_fun(indices, x, row, col):
  x = x.reshape((x.shape[0], -1, 128))

  def kernel(indices_ref, x_ref, o_ref, o_scratch, x_scratch):
    chunk = 8  # hardware specific value
    pltpu.sync_copy(x_ref.at[pl.ds(0, 8), ...], x_scratch)
    # @pl.loop(0, o_ref.shape[0], step=4)
    # def _(i):
    #   @pl.loop(0, o_ref.shape[-1], step=chunk)
    #   def _(j):
    #     j_, k_=  j // (128 // chunk), jax.lax.rem(j, (128 // chunk))
    #     o_scratch[i, pl.ds(j, chunk)] = 2 * x_scratch[0, j_, pl.ds(k_, chunk)]
    # pltpu.sync_copy(o_scratch, o_ref.at[pl.ds(0, 8), ...])

  out_shape = jax.ShapeDtypeStruct((x.shape[0], x.shape[-2] * x.shape[-1]), x.dtype)
  grid = (8,)

  in_specs = [
    pl.BlockSpec(
      (indices.shape[0] // grid[0],), lambda i: (i,),
      memory_space=pltpu.MemorySpace.VMEM
    ),
    pl.BlockSpec(memory_space=pltpu.MemorySpace.ANY),
  ]
  out_specs = pl.BlockSpec(memory_space=pltpu.MemorySpace.ANY)

  pallas_gather = pl.pallas_call(
    kernel,
    out_shape=out_shape, grid=grid, in_specs=in_specs, out_specs=out_specs,
    compiler_params=pltpu.CompilerParams(kernel_type=pltpu.KernelType.SC_VECTOR_SUBCORE),
    scratch_shapes=[
      pltpu.VMEM((8, x.shape[-1] * x.shape[-2]), x.dtype),
      pltpu.VMEM((8, x.shape[-2], x.shape[-1]), x.dtype)
    ]
  )
  out = pallas_gather(indices, x)
  return out

my_fun(indices, x, row=8, col=8).shape

JaxRuntimeError: INTERNAL: INVALID_ARGUMENT: SparseCore does not support transfers with relaxed ordering from local TileSpmem to local TileSpmem issued from TEC when both memories are on SparseCore.

at location: loc("my_fun.5":57:1)

The MLIR operation involved:
  "tpu.enqueue_dma"(%6, <<UNKNOWN SSA VALUE>>, %1) <{operandSegmentSizes = array<i32: 1, 0, 1, 1, 0, 0>, priority = 0 : i32, strict_ordering = false}> : (memref<8x56x128xi32, strided<[7168, 128, 1]>, #tpu.memory_space<any>>, memref<8x56x128xi32, #tpu.memory_space<vmem>>, memref<!tpu.dma_semaphore, #tpu.memory_space<semaphore_mem>>) -> ()
... additional diagnostics were skipped.

Please report a bug at: https://github.com/google/jax/issues/new?assignees=apaszke


In [ ]:
# %%
round_to = lambda x, r: pl.cdiv(max(x, r), r) * r

@partial(jax.jit, static_argnames=("row", "col"))
def my_fun(indices, x, row, col):
  x = x.reshape((x.shape[0], -1, 128))
  #row_window, col_window = round_to(min(row, num_steps), 16), round_to(min(x_cols, col), 128)
  row_window, col_window = round_to(min(row, num_steps), 16), col

  def kernel(indices_ref, x_ref, o_ref):
    #del indices_ref  # only used by the memory pipeline
    chunk = 16  # hardware specific value
    @pl.loop(0, o_ref.shape[-1], step=chunk)
    def _(i):
      #o_ref[:, pl.ds(i, chunk)] = x_ref[:, pl.ds(i, chunk)]
      @pl.loop(0, x_ref.shape[0], step=4)
      def _(j):
        j_, k_ = i // (128 // 16), jax.lax.rem(i, 128 // 16)
        # o_ref[pl.ds(j, 4), pl.ds(i, chunk)] = x_ref[pl.ds(j, 4), pl.ds(i, chunk)]
        #o_ref[pl.ds(j, 4), pl.ds(i, chunk)] = x_ref[pl.ds(j, 4), j_, pl.ds(k_, chunk)]
        # o_ref[pl.ds(j, 4), pl.ds(i, chunk)] = jnp.zeros((4, chunk), dtype=o_ref.dtype)

  #out_shape = x
  out_shape = jax.ShapeDtypeStruct((x.shape[0], x.shape[-2] * x.shape[-1]), x.dtype)
  grid = (pl.cdiv(indices.size, row_window), pl.cdiv(x.shape[-1], col_window))
  #grid = (pl.cdiv(indices.size, row_window),)

  in_specs = [
    pl.BlockSpec((row_window,), lambda i, j: i),
    # this indexed by performs the gather inside the memory pipeline itself
    plsc.BlockSpec((row_window, col_window, x.shape[-1]), lambda i, j: (0, j, 0), indexed_by=0, indexed_dim=0)
    #pl.BlockSpec((row_window, x.shape[1], x.shape[2]), lambda i: (i, 0, 0)),
  ]
  out_specs = pl.BlockSpec((row_window, col_window,), lambda i, j: (i, j))
  # out_specs = plsc.BlockSpec((row_window, out_shape.shape[-1],), lambda i: (0, 0), indexed_by=0, indexed_dim=0)

  pallas_gather = pl.pallas_call(
    kernel,
    out_shape=out_shape, grid=grid, in_specs=in_specs, out_specs=out_specs,
    compiler_params=pltpu.CompilerParams(dimension_semantics=("parallel", "arbitrary"), kernel_type=pltpu.KernelType.SC_VECTOR_SUBCORE),
    #compiler_params=pltpu.CompilerParams(dimension_semantics=("parallel",), kernel_type=pltpu.KernelType.SC_VECTOR_SUBCORE),
  )
  out = pallas_gather(indices, x)
  return out
  #assert jnp.linalg.norm(out - x[indices, :]) < 1e-3

In [ ]:
my_fun(indices, x, row=16, col=8)

In [ ]:
# %%
tune_jax.logger.setLevel("INFO")

hyperparams = {
  "row": [16, 32, 64, 128],
  "col": [128, 256, 512, 1024],
}
fn = tune_jax.tune(my_fun, hyperparams=hyperparams)
x_ = x.reshape((x.shape[0], -1, 128))
fn(indices, x_)
print(tune_jax.tabulate(fn))
fn2 = tune_jax.tune(jax.jit(lambda idx, x: x[idx, ...]))
fn2(indices, x_)
print(tune_jax.tabulate(fn2))

with jax.profiler.trace("/tmp/gather"):
  for _ in range(3):
    jax.block_until_ready(fn(indices, x_))
  for _ in range(3):
    jax.block_until_ready(fn2(indices, x_))

# Other

In [ ]:
# %%
num_steps, x_cols = 8 * 8 * 4096, 7168
x = jnp.arange(num_steps * x_cols).reshape(num_steps, x_cols).astype(jnp.bfloat16)  # random data
indices = jnp.arange(x.shape[0])

# %%
round_to = lambda x, r: pl.cdiv(max(x, r), r) * r

@partial(jax.jit, static_argnames=("row", "col"))
def my_fun(indices, x, row, col):
  x = x.reshape((x.shape[0], -1))
  row_window, col_window = round_to(min(row, num_steps), 16), round_to(min(x_cols, col), 128)

  def kernel(indices_ref, x_ref, o_ref):
    del indices_ref  # only used by the memory pipeline
    chunk = 16  # hardware specific value
    @pl.loop(0, x_ref.shape[-1], step=chunk)
    def _(i):
      #o_ref[:, pl.ds(i, chunk)] = x_ref[:, pl.ds(i, chunk)]
      @pl.loop(0, x_ref.shape[0], step=4)
      def _(j):
        o_ref[pl.ds(j, 4), pl.ds(i, chunk)] = x_ref[pl.ds(j, 4), pl.ds(i, chunk)]

  out_shape = x
  grid = (pl.cdiv(indices.size, row_window), pl.cdiv(x.shape[-1], col_window))

  in_specs = [
    pl.BlockSpec((row_window,), lambda i, j: i),
    # this indexed by performs the gather inside the memory pipeline itself
    plsc.BlockSpec((row_window, col_window,), lambda i, j: (0, j), indexed_by=0, indexed_dim=0)
  ]
  out_specs = pl.BlockSpec((row_window, col_window,), lambda i, j: (i, j))

  pallas_gather = pl.pallas_call(
    kernel,
    out_shape=out_shape, grid=grid, in_specs=in_specs, out_specs=out_specs,
    compiler_params=pltpu.CompilerParams(dimension_semantics=("parallel", "arbitrary"), kernel_type=pltpu.KernelType.SC_VECTOR_SUBCORE),
  )
  out = pallas_gather(indices, x)
  return out
  #assert jnp.linalg.norm(out - x[indices, :]) < 1e-3


# %%
tune_jax.logger.setLevel("INFO")

hyperparams = {
  "row": [16, 32, 64, 128],
  "col": [128, 256, 512, 1024],
}
fn = tune_jax.tune(my_fun, hyperparams=hyperparams)
x_ = x.reshape((x.shape[0], -1, 128))
fn(indices, x_)
print(tune_jax.tabulate(fn))
fn2 = tune_jax.tune(jax.jit(lambda idx, x: x[idx, ...]))
fn2(indices, x_)
print(tune_jax.tabulate(fn2))

with jax.profiler.trace("/tmp/gather"):
  for _ in range(3):
    jax.block_until_ready(fn(indices, x_))
  for _ in range(3):
    jax.block_until_ready(fn2(indices, x_))

In [ ]:
y1 = fn(indices, x_)
y2 = fn2(indices, x_)
print(y1.shape)
print(y2.shape)

In [ ]:
@jax.jit
def err_fn(y1, y2):
  return jnp.sum(jnp.abs(y2 - y1))

print(err_fn(y1, y2))

In [ ]:
y1.shape

In [ ]:
y2.shape

In [ ]:
(2 * 2 * x.size / 1640e9)

In [ ]:
@jax.jit
def concurrent(x, idx):
  y = my_fun(idx, x, row=16, col=512)
  max_row = 8192
  x_ = x[:max_row, :]
  z = x_.mT @ x_
  return y, z

_ = jax.block_until_ready(concurrent(x, indices))
with jax.profiler.trace("/tmp/concurrent"):
  for _ in range(3):
    _ = jax.block_until_ready(concurrent(x, indices))